In [ ]:
import gc
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
from numpy import arange
import pandas
import seaborn

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.checkpoint as cp
import torch.utils.data as ud
from torch.nn.utils import clip_grad as cg
from torch.optim.lr_scheduler import StepLR, LambdaLR
from torch.utils.data import DataLoader, Dataset, Sampler
from torch.utils.data.sampler import SubsetRandomSampler, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torchvision.transforms

from torchsummary import summary

import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import NeptuneLogger
from neptune.new.types import File

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

In [ ]:
# Check if its possible to use GPU/CUDA
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
    gc.collect()
    torch.cuda.empty_cache()
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.9,max_split_size_mb:128"
else:
    device = torch.device("cpu")
    print("Using CPU")
print(torch.version.cuda)
print(torch.version)
print(torchvision.version)

In [ ]:
params = {
    "train_batch_size": 32,
    "val_batch_size": 32,
    "test_batch_size": 32,
    "batch_norm_momentum": 0.2,
    "batch_norm_epsilon": 1e-3,
    "lr": 0.005,
    "Adam_beta1": 0.9,
    "Adam_beta2": 0.999,
    "Adam_epsilon": 1e-8,
    "weight_decay": 0.0,
    "dropout": 0.1,
    "max_epochs": 100,
    #"schedule": [30, 60, 90, 120],
    #"gamma": 0.1,
}

In [ ]:
# train_dir = "E:/pdm/ALASKA_SUNIWARD_FILTERED/train"
# val_dir = "E:/pdm/ALASKA_SUNIWARD_FILTERED/val"
# test_dir = "E:/pdm/ALASKA_SUNIWARD_FILTERED/test"

# train_dir = "E:/pdm/BossBase_Alaska_dataset/train"
# val_dir = "E:/pdm/BossBase_Alaska_dataset/val"
# test_dir = "E:/pdm/BossBase_Alaska_dataset/test"

# train_dir = "E:/pdm/BossBase_Alaska_dataset/train_filtered"
# val_dir = "E:/pdm/BossBase_Alaska_dataset/val_filtered"
# test_dir = "E:/pdm/BossBase_Alaska_dataset/test_filtered"

# train_dir = "E:/pdm/BOSS_SUNIWARD/train"
# val_dir = "E:/pdm/BOSS_SUNIWARD/val"
# test_dir = "E:/pdm/BOSS_SUNIWARD/test"

In [ ]:
# train_dir = "E:/pdm/BOSS_SUNIWARD/train"
# val_dir = "E:/pdm/BOSS_SUNIWARD/val"
# test_dir = "E:/pdm/BOSS_SUNIWARD/test"

train_dir = "E:/pdm/BossBase_GBRAS/train"
val_dir = "E:/pdm/BossBase_GBRAS/val"
test_dir = "E:/pdm/BossBase_GBRAS/test"
# is_torchvision_installed = True
# try:
#     import torchvision
# except:
#     is_torchvision_installed = False

# class BalancedBatchSampler(torch.utils.data.sampler.Sampler):
#     def __init__(self, dataset, labels=None):
#         self.labels = labels
#         self.dataset = dict()
#         self.balanced_max = 0
#         # Save all the indices for all the classes
#         for idx in range(0, len(dataset)):
#             label = self._get_label(dataset, idx)
#             if label not in self.dataset:
#                 self.dataset[label] = list()
#             self.dataset[label].append(idx)
#             self.balanced_max = len(self.dataset[label]) \
#                 if len(self.dataset[label]) > self.balanced_max else self.balanced_max
        
#         # Oversample the classes with fewer elements than the max
#         for label in self.dataset:
#             while len(self.dataset[label]) < self.balanced_max:
#                 self.dataset[label].append(random.choice(self.dataset[label]))
#         self.keys = list(self.dataset.keys())
#         self.currentkey = 0
#         self.indices = [-1]*len(self.keys)

#     def __iter__(self):
#         while self.indices[self.currentkey] < self.balanced_max - 1:
#             self.indices[self.currentkey] += 1
#             yield self.dataset[self.keys[self.currentkey]][self.indices[self.currentkey]]
#             self.currentkey = (self.currentkey + 1) % len(self.keys)
#         self.indices = [-1]*len(self.keys)
    
#     def _get_label(self, dataset, idx, labels = None):
#         if self.labels is not None:
#             return self.labels[idx].item()
#         else:
#             # Trying guessing
#             dataset_type = type(dataset)
#             if is_torchvision_installed and dataset_type is torchvision.datasets.MNIST:
#                 return dataset.train_labels[idx].item()
#             elif is_torchvision_installed and dataset_type is torchvision.datasets.ImageFolder:
#                 return dataset.imgs[idx][1]
#             else:
#                 raise Exception("You should pass the tensor of labels to the constructor as second argument")

#     def __len__(self):
#         return self.balanced_max*len(self.keys)

class Data(pl.LightningDataModule):
    def __init__(self):
        super().__init__()
        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.ToTensor()
        ])
        self.num_workers = 0
        self.seed_value = 42
        torch.manual_seed(self.seed_value)

    def prepare_data(self):
        self.train = ImageFolder(root=train_dir, transform=self.transform)
        self.val = ImageFolder(root=val_dir, transform=self.transform)
        self.test = ImageFolder(root=test_dir, transform=self.transform)

    def setup(self, stage: str):
        if stage == "fit":
            # Shuffle the indices of the dataset with a fixed seed
            train_indices = list(range(len(self.train)))
            random.shuffle(train_indices)
            # Create a dataloader with shuffled indices
            self.train_set = DataLoader(self.train, batch_size=params["train_batch_size"], shuffle=False, sampler=SubsetRandomSampler(train_indices), num_workers=self.num_workers, pin_memory=True, drop_last=True) 

            val_indices = list(range(len(self.val)))
            random.shuffle(val_indices)
            self.val_set = DataLoader(self.val, batch_size=params["val_batch_size"], shuffle=False, sampler=SubsetRandomSampler(val_indices), num_workers=self.num_workers, pin_memory=True, drop_last=True)
        if stage == "test":
            test_indices = list(range(len(self.test)))
            random.shuffle(test_indices)
            self.test_set = DataLoader(self.test, batch_size=params["test_batch_size"], shuffle=False, sampler=SubsetRandomSampler(test_indices), num_workers=self.num_workers, pin_memory=True, drop_last=True) 

    def train_dataloader(self):
        return self.train_set

    def val_dataloader(self):
        return self.val_set

    def test_dataloader(self):
        return self.test_set
    
data_module = Data()

In [ ]:
class Abs(nn.Module):
    def __init__(self):
        super(Abs, self).__init__()

    def forward(self, x):
        output = torch.abs(x)
        return output
    
class Trunc(nn.Module):
    def __init__(self, threshold):
        super(Trunc, self).__init__()
        self.threshold = threshold

    def forward(self, x):
        output = torch.clamp(x, min = -self.threshold, max = self.threshold)
        return output
    
class ScaleLayer(nn.Module):
    def __init__(self, num_features):
        super(ScaleLayer, self).__init__()
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        return x * self.gamma.view(1, -1, 1, 1) + self.beta.view(1, -1, 1, 1)


In [ ]:
T3 = 3
def Tanh3(x):
    tanh3 = torch.tanh(x)*T3
    return tanh3

### Define CNN architecture ###
class GBRAS(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.labels = ["cover", "stego"]
        self.loss_fn = nn.CrossEntropyLoss()
        self.dropout = nn.Dropout(params["dropout"])
        self.train_ytrue = []
        self.train_ypred = []
        self.val_ytrue = []
        self.val_ypred = []
        self.test_ytrue = []
        self.test_ypred = []

        self.elu = nn.ELU()
   
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=30, kernel_size=5, stride=1, padding=2) 
        #self.conv1.weight = nn.Parameter(srm_weights.float())
        # self.conv1.bias = nn.Parameter(biasSRM.float())
        self.bn1 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"],eps=params["batch_norm_epsilon"],
                                   affine=True)
        
        self.depthwise_conv1 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=1, groups=3) 
        self.separable_conv1 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=3, padding=1) 
        self.bn2 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.depthwise_conv2 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=1, groups=3) 
        self.separable_conv2 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True) 

        self.conv2 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.conv3 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=3, stride=1, padding=1)
        self.bn5 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.conv4 = nn.Conv2d(in_channels=30, out_channels=60, kernel_size=3, stride=1, padding=1)
        self.bn6 = nn.BatchNorm2d(num_features=60, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.depthwise_conv3 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=1, groups=3)
        self.separable_conv3 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=3, padding=1)
        self.bn7 = nn.BatchNorm2d(num_features=60, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.depthwise_conv4 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=1, groups=3)
        self.separable_conv4 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=3, padding=1)
        self.bn8 = nn.BatchNorm2d(num_features=60, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)

        self.conv5 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=3, stride=1, padding=1)
        self.bn9 = nn.BatchNorm2d(num_features=60, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                   affine=True)
          
        self.conv6 = nn.Conv2d(in_channels=60, out_channels=60, kernel_size=3, stride=1, padding=1)
        self.bn10 = nn.BatchNorm2d(num_features=60, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                    affine=True)

        self.conv7 = nn.Conv2d(in_channels=60, out_channels=30, kernel_size=3, stride=1, padding=1)
        self.bn11 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                    affine=True)

        self.conv8 = nn.Conv2d(in_channels=30, out_channels=30, kernel_size=1, stride=1, padding=1)
        self.bn12 = nn.BatchNorm2d(num_features=30, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                    affine=True)
        
        self.conv9 = nn.Conv2d(in_channels=30, out_channels=2, kernel_size=1, stride=1, padding=1)
        self.bn13 = nn.BatchNorm2d(num_features=2, momentum=params["batch_norm_momentum"], eps=params["batch_norm_epsilon"],
                                    affine=True)

    def init_weights(self, m):
        if type(m) == (nn.Linear or nn.Conv2d):
            nn.init.xavier_uniform_(m.weight)
        
    def forward(self, layers):
        layers = self.elu(self.conv1(layers))
        layers = self.dropout(layers)
        layers1 = self.bn1(layers)
        #layers = Tanh3(layers)
        
        layers = self.depthwise_conv1(layers1)
        layers = self.elu(self.separable_conv1(layers))
        layers = self.dropout(layers)
        layers = self.bn2(layers)

        layers = self.depthwise_conv2(layers)
        layers = self.elu(self.separable_conv2(layers))
        layers = self.dropout(layers)
        layers2 = self.bn3(layers)

        skip1 = torch.add(layers1, layers2)

        layers = self.elu(self.conv2(skip1))
        layers = self.dropout(layers)
        layers = self.bn4(layers)
       
        layers = self.elu(self.conv3(layers))
        layers = self.dropout(layers)
        layers = self.bn5(layers)

        layers = nn.AvgPool2d(kernel_size=2, stride=2)(layers)
        layers = self.elu(self.conv4(layers))
        layers = self.dropout(layers)
        layers3 = self.bn6(layers)

        layers = self.depthwise_conv3(layers3)
        layers = self.elu(self.separable_conv3(layers))
        layers = self.dropout(layers)
        layers = self.bn7(layers)

        layers = self.depthwise_conv4(layers)
        layers = self.elu(self.separable_conv4(layers))
        layers = self.dropout(layers)
        layers4 = self.bn8(layers)

        skip2 = torch.add(layers3, layers4)
        
        layers = self.elu(self.conv5(skip2))
        layers = self.dropout(layers)
        layers = self.bn9(layers)

        layers = nn.AvgPool2d(kernel_size=2, stride=2)(layers)
        layers = self.elu(self.conv6(layers))
        layers = self.dropout(layers)
        layers = self.bn10(layers)
      

        layers = nn.AvgPool2d(kernel_size=2, stride=2)(layers)
        layers = self.elu(self.conv7(layers))
        layers = self.dropout(layers)
        layers = self.bn11(layers)
    

        layers = nn.AvgPool2d(kernel_size=2, stride=2)(layers)

        layers = self.elu(self.conv8(layers))
        layers = self.dropout(layers)
        layers = self.bn12(layers)

        layers = self.elu(self.conv9(layers))
        layers = self.dropout(layers)
        layers = self.bn13(layers)

        layers = F.adaptive_avg_pool2d(layers, (1, 1))
        layers = layers.view(-1, 2)
        return layers
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), lr=params["lr"], betas=(params["Adam_beta1"], params["Adam_beta2"]),
                eps=params["Adam_epsilon"], weight_decay=params["weight_decay"]
        )
        #scheduler = optim.lr_scheduler.MultiStepLR(optimizer=optimizer, milestones=params["schedule"], gamma=params["gamma"])

        return [optimizer]#, [scheduler]

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("train/loss", loss, on_step=False, on_epoch=True)

        y_true = y
        y_pred = y_hat.argmax(dim=1)
    
        acc = accuracy_score(y_true.cpu().numpy(), y_pred.cpu().numpy())
        self.log("train/acc", acc, on_step=False, on_epoch=True)
        
        #self.train_ytrue.append(y_true)
        #self.train_ypred.append(y_pred)
        return loss

    # def on_train_epoch_end(self):s
    #     y_true = torch.stack(self.train_ytrue).cpu().numpy()
    #     y_pred = torch.stack(self.train_ypred).cpu().numpy()

    #     prec = precision_score(y_true, y_pred, average="micro")
    #     recall = recall_score(y_true, y_pred, average="micro")
    #     f1 = f1_score(y_true, y_pred, average="micro")
    #     roc_auc = roc_auc_score(y_true, y_pred, average="micro")

    #     self.log("train/prec", prec, on_step=False,on_epoch=True)
    #     self.log("train/recall", recall, on_step=False, on_epoch=True)
    #     self.log("train/f1", f1, on_step=False, on_epoch=True)
    #     self.log("train/roc_auc", roc_auc,on_step=False, on_epoch=True)

    #     self.train_ytrue.clear()
    #     self.train_ypred.clear()


    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("val/loss", loss, on_step=False, on_epoch=True)
       
        y_true = y
        y_pred = y_hat.argmax(dim=1)
        acc = accuracy_score(y_true.cpu().numpy(), y_pred.cpu().numpy())
        self.log("val/acc", acc, on_step=False, on_epoch=True)

        self.val_ytrue.append(y_true)
        self.val_ypred.append(y_pred)

        return loss

    def on_validation_epoch_end(self):
        y_true = torch.stack(self.val_ytrue).cpu().numpy()
        y_pred = torch.stack(self.val_ypred).cpu().numpy()

        prec = precision_score(y_true, y_pred, average="micro")
        recall = recall_score(y_true, y_pred, average="micro")
        f1 = f1_score(y_true, y_pred, average="micro")
        roc_auc = roc_auc_score(y_true, y_pred, average="micro")

        self.log("val/prec", prec, on_step=False, on_epoch=True)
        self.log("val/recall", recall, on_step=False, on_epoch=True)
        self.log("val/f1", f1, on_step=False, on_epoch=True)
        self.log("val/roc_auc", roc_auc,on_step=False, on_epoch=True)

        y_true_labels = np.argmax(y_true, axis=1)
        y_pred_labels = np.argmax(y_pred, axis=1)
        cm = confusion_matrix(y_true_labels, y_pred_labels, labels=[0, 1], normalize='true')
        df_cfm = pandas.DataFrame(cm, index=self.labels, columns=self.labels)
        fig, ax = plt.subplots(figsize=(10, 7))
        seaborn.heatmap(df_cfm, annot=True, cmap='Blues', fmt='.2f', ax=ax)
        neptune_logger.experiment["val_cm"].append(File.as_image(fig))
            
        self.val_ytrue.clear()
        self.val_ypred.clear()
    
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log("test/loss", loss, on_step=True, on_epoch=True)

        y_true = y
        y_pred = y_hat.argmax(dim=1)
        acc = accuracy_score(y_true.cpu().numpy(), y_pred.cpu().numpy())
        self.log("test/acc", acc, on_step=True, on_epoch=True)

        self.test_ytrue.append(y_true)
        self.test_ypred.append(y_pred)

        return loss

    def on_test_epoch_end(self):
        y_true = torch.stack(self.test_ytrue).cpu().numpy()
        y_pred = torch.stack(self.test_ypred).cpu().numpy()

        prec = precision_score(y_true, y_pred, average="micro")
        recall = recall_score(y_true, y_pred, average="micro")
        f1 = f1_score(y_true, y_pred, average="micro")
        roc_auc = roc_auc_score(y_true, y_pred, average="micro")

        self.log("test/prec", prec, on_step=False, on_epoch=True)
        self.log("test/recall", recall, on_step=False, on_epoch=True)
        self.log("test/f1", f1, on_step=False, on_epoch=True)
        self.log("test/roc_auc", roc_auc, on_step=False, on_epoch=True)

        y_true_labels = np.argmax(y_true, axis=1)
        y_pred_labels = np.argmax(y_pred, axis=1)
        cm = confusion_matrix(y_true_labels, y_pred_labels, labels=[0, 1], normalize='true')
        df_cfm = pandas.DataFrame(cm, index=self.labels, columns=self.labels)
        fig, ax = plt.subplots(figsize=(10, 7))
        seaborn.heatmap(df_cfm, annot=True, cmap='Blues', fmt='.2f', ax=ax)
        neptune_logger.experiment["test_cm"].append(File.as_image(fig))
            
        self.test_ytrue.clear()
        self.test_ypred.clear()
    

In [ ]:
# Create NeptunLogger
neptune_logger = NeptuneLogger(
    api_key = os.environ.get('API_KEY'),
    project="eqebenezer/gbras",
    prefix="experiment",
    tags=["BossBase(252), matlab filtered"],
    log_model_checkpoints=True, 
    capture_hardware_metrics=False,
    source_files="GBRAS.ipynb"
)

run_id = neptune_logger.run["sys/id"].fetch()
folder_name = run_id
log_path = "E:/pdm/GBRAS"

path = os.path.join(log_path, folder_name)
os.makedirs(path, exist_ok=True)

# Create learning rate logger
lr_logger = LearningRateMonitor(logging_interval="epoch")

# Create model checkpointing object
model_checkpoint = ModelCheckpoint(
    dirpath=path,
    mode="max",
    monitor="val/acc",
    save_weights_only=False,
    save_top_k=15,
    save_last=True,
    every_n_epochs=5,
)

# Initialize a trainer and pass neptune_logger
trainer = pl.Trainer(
    default_root_dir=path,
    logger=neptune_logger,
    callbacks=[lr_logger, model_checkpoint],
    log_every_n_steps=150,
    accelerator="cuda",
    devices=1,
    max_epochs=params["max_epochs"],
    val_check_interval=1.0,
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

In [ ]:
model = GBRAS()

In [ ]:
# Log model summary
neptune_logger.log_model_summary(model=model, max_depth=-1)

# Log hyperparameters
neptune_logger.log_hyperparams(params=params)

In [ ]:
# Train
trainer.fit(model, datamodule=data_module)

In [ ]:
# Test
trainer.test(model, datamodule=data_module)

In [ ]:
neptune_logger.experiment.stop()

In [ ]:
torch.cuda.empty_cache()
gc.collect()